# Boarding Pass Verification System — FRA → TAS
**Loyiha:** Frankfurt → Tashkent parvozi uchun boarding pass tekshirish tizimi

| Bosqich | Tavsif |
|---------|--------|
| 1 | Yo'lovchilar bazasi (FRA → TAS) |
| 2 | Preprocessing (CLAHE, deskew, adaptive threshold) |
| 3 | EasyOCR — ticket formatlaridan ma'lumot o'qish |
| 4 | Yo'lovchi ma'lumotlarini baza bilan solishtirish |
| 5 | Gradio UI: baza ko'rinishi + ticket tekshirish |

## Step 1 — O'rnatish
Kerakli kutubxonalarni o'rnatish (birinchi marta ishga tushirishda kerak)

In [1]:
!pip install python-doctr, easyocr gradio opencv-python pillow pandas -q


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\xolmu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'python-doctr,': Expected semicolon (after name with no version specifier) or end
    python-doctr,
                ^


##  Step 2 — Importlar
Barcha kerakli kutubxonalarni import qilish

In [2]:
import easyocr
import cv2
import numpy as np
import gradio as gr
import pandas as pd
from PIL import Image, ImageEnhance
import re
from difflib import SequenceMatcher

print('✅ Barcha kutubxonalar yuklandi!')

✅ Barcha kutubxonalar yuklandi!


##  Step 3 — Yo'lovchilar Bazasi
HY502 parvozi uchun yo'lovchilar ro'yxati: FRa (Fransiya) → TAS (Tashkent)

In [3]:
# Parvoz ma'lumotlari: HY502 | FRA → TAS | 15 JUN 2025 | GATE B7 | 13:40

PASSENGERS_DATA = [
    {'id': 'P001', 'name': 'ISMOILOV AZIZBEK',       'passport': 'AA1234567', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '12A', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P002', 'name': 'RUSTAMOVA MADINA',       'passport': 'AA2345678', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '12B', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P003', 'name': 'SODIQOV SHERZOD',        'passport': 'AB3456789', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '14C', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P004', 'name': 'HAMIDOVA DILNOZA',       'passport': 'AB4567890', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '14D', 'gate': 'B7', 'status': 'BOARDING'},
    {'id': 'P005', 'name': 'MAMATQULOV ULUGBEK',     'passport': 'AC5678901', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '16A', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P006', 'name': 'SHARIPOV OYBEK',         'passport': 'AC6789012', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '16B', 'gate': 'B7', 'status': 'BOARDING'},
    {'id': 'P007', 'name': 'ABDURAHMONOVA NIGINA',   'passport': 'AD7890123', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '18C', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P008', 'name': 'JURAYEV ASADBEK',        'passport': 'AD8901234', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '18D', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P009', 'name': 'KAMILOV SHAHZOD',        'passport': 'AE9012345', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '20A', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P010', 'name': 'ERKINOVA MOHIRA',        'passport': 'AE0123456', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '20B', 'gate': 'B7', 'status': 'BOARDING'},
    {'id': 'P011', 'name': 'NABIYEV JAHONGIR',       'passport': 'AF1234567', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '22C', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P012', 'name': 'SATTOROVA GULNOZA',      'passport': 'AF2345678', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '22D', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P013', 'name': 'TOJIBOYEV MUHAMMADALI',  'passport': 'AG3456789', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '24A', 'gate': 'B7', 'status': 'BOARDING'},
    {'id': 'P014', 'name': 'RAJABOVA ZARNIGOR',      'passport': 'AG4567890', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '24B', 'gate': 'B7', 'status': 'CHECKED IN'},
    {'id': 'P015', 'name': 'XOLMUMINOV QUTBIDDIN',   'passport': 'AH5678901', 'flight': 'HY502', 'from': 'FRA', 'to': 'TAS', 'date': '15 JUN 2025', 'seat': '26C', 'gate': 'B7', 'status': 'CHECKED IN'},
]

print(f"Jami yo'lovchilar: {len(PASSENGERS_DATA)}")

Jami yo'lovchilar: 15


In [4]:
# DataFrame yaratish va ko'rsatish
df_passengers = pd.DataFrame(PASSENGERS_DATA)

print('✈️  Flight: HY502  |  FRA → TAS  |  15 JUN 2025')
print()
print(df_passengers[['id', 'name', 'seat', 'status']].to_string(index=False))

✈️  Flight: HY502  |  FRA → TAS  |  15 JUN 2025

  id                  name seat     status
P001      ISMOILOV AZIZBEK  12A CHECKED IN
P002      RUSTAMOVA MADINA  12B CHECKED IN
P003       SODIQOV SHERZOD  14C CHECKED IN
P004      HAMIDOVA DILNOZA  14D   BOARDING
P005    MAMATQULOV ULUGBEK  16A CHECKED IN
P006        SHARIPOV OYBEK  16B   BOARDING
P007  ABDURAHMONOVA NIGINA  18C CHECKED IN
P008       JURAYEV ASADBEK  18D CHECKED IN
P009       KAMILOV SHAHZOD  20A CHECKED IN
P010       ERKINOVA MOHIRA  20B   BOARDING
P011      NABIYEV JAHONGIR  22C CHECKED IN
P012     SATTOROVA GULNOZA  22D CHECKED IN
P013 TOJIBOYEV MUHAMMADALI  24A   BOARDING
P014     RAJABOVA ZARNIGOR  24B CHECKED IN
P015  XOLMUMINOV QUTBIDDIN  26C CHECKED IN


##  Step 4a — Preprocessing: deskew()
Qiyshiq (egilgan) rasmlarni tekislash funksiyasi

In [5]:
def deskew(gray):
    """Qiyshiq rasmni tekislaydi (burish burchagini topib, tuzatadi)"""
    coords = np.column_stack(np.where(gray < 128))
    if len(coords) < 10:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.5:   # 0.5 gradusdan kam qiyiqlikni o'tkazib yuborish
        return gray
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)

print('✅ deskew() tayyor!')

✅ deskew() tayyor!


##  Step 4b — Preprocessing: preprocess_advanced()
Rasmni 4 xil usulda qayta ishlaydi: CLAHE, Otsu, Adaptive threshold, Morph

In [6]:
def preprocess_advanced(image):
    """Bir nechta preprocessing versiyasini qaytaradi: original, clahe, otsu, adaptive, morph"""
    img_np  = np.array(image)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    # Kichik rasmlarni kattalashtirish (OCR uchun min 1000px)
    h, w = img_bgr.shape[:2]
    if max(h, w) < 1000:
        scale   = 1000 / max(h, w)
        img_bgr = cv2.resize(img_bgr, None, fx=scale, fy=scale,
                             interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = deskew(gray)

    # Versiya 1: CLAHE — kontrast oshirish
    clahe    = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    v1_clahe = clahe.apply(gray)

    # Versiya 2: Gaussian blur + Otsu threshold
    blurred  = cv2.GaussianBlur(gray, (3, 3), 0)
    _, v2_otsu = cv2.threshold(blurred, 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Versiya 3: Adaptive threshold — notekis yoritilgan ticketlar uchun
    v3_adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 10
    )

    # Versiya 4: Morph — shovqin yo'qotish
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    v4_morph = cv2.morphologyEx(v2_otsu, cv2.MORPH_OPEN, kernel)

    return img_bgr, v1_clahe, v2_otsu, v3_adaptive, v4_morph

print('✅ preprocess_advanced() tayyor!')

✅ preprocess_advanced() tayyor!


##  Step 5a — EasyOCR Reader
EasyOCR modelini yuklash (birinchi marta model yuklab olinadi)

In [7]:
reader = easyocr.Reader(['en','uz'], gpu=False, verbose=False)
print('✅ EasyOCR reader tayyor!')

✅ EasyOCR reader tayyor!


##  Step 5b — Multi-version OCR
Barcha preprocessing versiyalarida OCR o'tkazib, natijalarni birlashtiradi

In [8]:
def run_ocr_multi(img_bgr, v1_clahe, v2_otsu, v3_adaptive, v4_morph, conf_threshold=0.15):
    """Barcha 5 preprocessing versiyasida OCR ishlatadi, eng yaxshi natijani saqlaydi."""
    sources = [img_bgr, v1_clahe, v2_otsu, v3_adaptive, v4_morph]

    best_texts = {}  # key=normalized_text, value=(bbox, text, conf)

    for src in sources:
        results = reader.readtext(src, detail=1)
        for (bbox, text, conf) in results:
            if conf < conf_threshold:
                continue
            # Normalizatsiya: katta harf, bo'shliqsiz
            key = re.sub(r'\s+', '', text.upper())
            if key not in best_texts or best_texts[key][2] < conf:
                best_texts[key] = (bbox, text, conf)

    # y, x tartibida saralash (yuqoridan pastga, chapdan o'ngga)
    merged = list(best_texts.values())
    merged.sort(key=lambda x: (x[0][0][1], x[0][0][0]))
    return merged

print('✅ run_ocr_multi() tayyor!')

✅ run_ocr_multi() tayyor!


##  Step 6a — Konstantalar: IATA Kodlari va Skip So'zlar
Ma'lum aeroportlar ro'yxati va boarding pass-da ism emas deb hisoblanadigan so'zlar

In [9]:
# Barcha asosiy aeroportlar IATA kodlari
KNOWN_IATA = {
    'ICN', 'GMP', 'JFK', 'LAX', 'LHR', 'CDG', 'NRT', 'PVG', 'SIN', 'DXB',
    'HKG', 'BKK', 'SYD', 'ORD', 'ATL', 'DFW', 'CGK', 'SVO', 'DME', 'FCO',
    'AMS', 'FRA', 'MAD', 'BCN', 'IST', 'DEL', 'BOM', 'PEK', 'SHA', 'KUL',
    'MNL', 'GRU', 'TAS', 'SKD', 'NMA', 'UGC', 'FEG', 'KMQ', 'BHK', 'NCU',
    'DYU', 'FRU', 'ALA', 'TSE', 'GYD', 'TBS', 'EVN', 'VIE', 'ZRH', 'GVA',
    'CPH', 'ARN', 'HEL', 'OSL', 'LIS', 'PRG', 'BUD', 'WAW', 'BRU', 'MUC',
}

# Boarding pass-da uchrashi mumkin bo'lgan lekin ism EMAS so'zlar
SKIP_WORDS = {
    'BOARDING', 'PASS', 'BOARDING PASS', 'PASSENGER', 'DESTINATION',
    'FLIGHT', 'NUMBER', 'DATETIME', 'DATE', 'TIME', 'SEAT', 'GATE',
    'FROM', 'DEPARTURE', 'ARRIVAL', 'TICKET', 'DISCOUNT', 'CLASS',
    'ECONOMY', 'BUSINESS', 'FIRST', 'AIR', 'AIRLINES', 'AIRWAYS',
    'TERMINAL', 'INTERNATIONAL', 'AIRPORT', 'CHECK', 'CHECKED',
}

print(f'IATA kodlar: {len(KNOWN_IATA)} ta')
print(f'Skip words: {len(SKIP_WORDS)} ta')

IATA kodlar: 60 ta
Skip words: 29 ta


##  Step 6b — extract_ticket_info()
OCR natijalaridan flight, sana, o'rindiq, gate, marshut va ism ajratib oladi

In [10]:
def extract_ticket_info(ocr_results):
    """OCR natijalaridan barcha ticket ma'lumotlarini ajratadi."""
    lines        = [text for (_, text, _) in ocr_results]
    full_text    = ' '.join(lines)
    full_upper   = full_text.upper()

    info = {
        'raw_text': full_text,
        'name'    : 'Not found',
        'flight'  : 'Not found',
        'date'    : 'Not found',
        'seat'    : 'Not found',
        'gate'    : 'Not found',
        'from'    : 'Not found',
        'to'      : 'Not found',
    }

    # Flight raqami: HY502, KE123, OZ456
    flight_m = re.search(r'\b([A-Z]{2,3}\s?\d{2,5})\b', full_upper)
    if flight_m:
        info['flight'] = flight_m.group(1).replace(' ', '')

    # Sana: 15 JUN 2025 | 2025-06-15 | 15/06/2025 | 15JUN25
    date_patterns = [
        r'\d{1,2}\s(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\s\d{2,4}',
        r'\d{4}[-/]\d{2}[-/]\d{2}',
        r'\d{2}[-/]\d{2}[-/]\d{4}',
        r'\d{1,2}(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\d{2,4}',
    ]
    for pat in date_patterns:
        m = re.search(pat, full_upper)
        if m:
            info['date'] = m.group(0)
            break

    # O'rindiq: SEAT 12A yoki shunchaki 12A
    seat_kw = re.search(r'(?:SEAT|KURSI)\s*[:\-]?\s*(\d{1,3}[A-HJ-Z]?)', full_upper)
    if seat_kw:
        info['seat'] = seat_kw.group(1)
    else:
        seat_plain = re.search(r'\b(\d{1,2}[A-F])\b', full_upper)
        if seat_plain:
            info['seat'] = seat_plain.group(1)

    # Gate: GATE B7 yoki shunchaki B7
    gate_kw = re.search(r'(?:GATE|DARVOZA)\s*[:\-]?\s*([A-Z]?\d{1,3})', full_upper)
    if gate_kw:
        info['gate'] = gate_kw.group(1)
    else:
        gate_plain = re.search(r'\b([A-Z]\d{1,2})\b', full_upper)
        if gate_plain:
            info['gate'] = gate_plain.group(1)

    # Marshut: ICN → TAS yoki FROM/TO kalit so'zlari
    route = re.search(r'([A-Z]{3})\s*[-=\u2014\u2192>]+\s*([A-Z]{3})', full_upper)
    if route:
        info['from'] = route.group(1)
        info['to']   = route.group(2)
    else:
        found_iata   = [a for a in re.findall(r'\b([A-Z]{3})\b', full_upper) if a in KNOWN_IATA]
        flight_code  = info['flight'][:2] if info['flight'] != 'Not found' else ''
        found_iata   = [a for a in found_iata if a != flight_code]
        if len(found_iata) >= 2:
            info['from'] = found_iata[0]
            info['to']   = found_iata[1]
        elif len(found_iata) == 1:
            info['from'] = found_iata[0]
        if info['from'] == 'Not found':
            m_from = re.search(r'(?:FROM|DEPARTURE|DEP)\s*[:\-]?\s*([A-Z]{3,})', full_upper)
            if m_from:
                info['from'] = m_from.group(1)[:3]
        if info['to'] == 'Not found':
            m_to = re.search(r'(?:TO|DESTINATION|ARRIVAL|ARR)\s*[:\-]?\s*([A-Z]{3,})', full_upper)
            if m_to:
                info['to'] = m_to.group(1)[:3]

    # Ism: PASSENGER kalit so'z, yoki MR/MRS, yoki ikki katta so'z
    name_kw = re.search(r'(?:PASSENGER|NAME|PAX)\s*[:\-]?\s*([A-Z]{2,}\s+[A-Z]{2,})', full_upper)
    if name_kw:
        info['name'] = name_kw.group(1).title()
    else:
        name_mr = re.search(r'\b(?:MR|MRS|MS|DR)\.?\s+([A-Z]+\s+[A-Z]+)', full_upper)
        if name_mr:
            info['name'] = name_mr.group(1).title()
        else:
            for m in re.finditer(r'\b([A-Z]{3,}\s+[A-Z]{3,})\b', full_upper):
                candidate = m.group(1)
                parts     = candidate.split()
                if not any(p in SKIP_WORDS for p in parts):
                    if not re.search(r'\d', candidate):
                        info['name'] = candidate.title()
                        break

    return info

print("✅ extract_ticket_info() tayyor!")

✅ extract_ticket_info() tayyor!


##  Step 7a — name_similarity()
Ikki ismning o'xshashlik foizini hisoblaydigan yordamchi funksiya

In [11]:
def name_similarity(a, b):
    """Ikki ismning o'xshashlik koeffitsiyenti: 0.0 dan 1.0 gacha"""
    return SequenceMatcher(None, a.upper(), b.upper()).ratio()

# Test
print(name_similarity('NABIYEV JAHONGIR', 'NABIYEV JAHONGIR'))  # 1.0 — aniq
print(name_similarity('NABIYEW JAHONGIR', 'NABIYEV JAHONGIR'))  # ~0.93 — OCR xatosi
print(name_similarity('NABIYEV JAHONGIR',   'MARDIEV NAVRUZ'))  # ~0.46 — boshqa odam

1.0
0.9375
0.4666666666666667


##  Step 7b — find_passenger()
3 usulda yo'lovchini bazada qidiradi: aniq ism, fuzzy ism, o'rindiq+flight

In [12]:
def find_passenger(info):
    """OCR ma'lumotlari asosida yo'lovchini bazada qidiradi."""
    flight = info['flight'].upper().replace(' ', '')
    name   = info['name'].upper()
    seat   = info['seat'].upper()

    # 1-usul: Ism + flight bo'yicha aniq moslik
    if name != 'NOT FOUND' and flight != 'NOT FOUND':
        mask = (
            (df_passengers['flight'].str.upper() == flight) &
            (df_passengers['name'].str.upper()   == name)
        )
        if mask.any():
            return df_passengers[mask].iloc[0], 'exact_name'

    # 2-usul: Fuzzy name matching (OCR xatolarini hisobga olish)
    if name != 'NOT FOUND':
        best_score, best_row = 0, None
        for _, row in df_passengers.iterrows():
            if flight != 'NOT FOUND' and row['flight'].upper() != flight:
                continue
            score = name_similarity(name, row['name'])
            if score > best_score:
                best_score, best_row = score, row
        if best_score >= 0.75:   # 75% o'xshashlik yetarli
            return best_row, 'fuzzy_name'

    # 3-usul: O'rindiq + flight bo'yicha
    if seat != 'NOT FOUND' and flight != 'NOT FOUND':
        mask = (
            (df_passengers['flight'].str.upper() == flight) &
            (df_passengers['seat'].str.upper()   == seat)
        )
        if mask.any():
            return df_passengers[mask].iloc[0], 'seat_flight'

    return None, 'not_found'

print('✅ find_passenger() tayyor!')

✅ find_passenger() tayyor!


##  Step 7c — verify_passenger()
Yo'lovchini topib, barcha maydonlarni solishtiradi: ✅ mos, ❌ mos emas, ⚠️ topilmadi

In [13]:
def verify_passenger(info):
    """To'liq tekshirish: yo'lovchi topildimi va maydonlar mos kelishini tekshiradi."""
    passenger, match_type = find_passenger(info)

    if passenger is None:
        return {
            'status'      : 'DENIED',
            'icon'        : '❌',
            'message'     : "Yo'lovchi bazada topilmadi!",
            'match_type'  : 'not_found',
            'passenger'   : None,
            'field_checks': {},
            'warnings'    : [],
        }

    def chk(ocr_val, db_val):
        if ocr_val in ('Not found', 'NOT FOUND', ''):
            return '⚠️'  # Ma'lumot topilmadi
        return '✅' if ocr_val.strip().upper() == db_val.strip().upper() else '❌'

    field_checks = {
        'Name'  : (chk(info['name'],   passenger['name']),   info['name'],   passenger['name']),
        'Flight': (chk(info['flight'], passenger['flight']), info['flight'], passenger['flight']),
        'Date'  : (chk(info['date'],   passenger['date']),   info['date'],   passenger['date']),
        'Seat'  : (chk(info['seat'],   passenger['seat']),   info['seat'],   passenger['seat']),
        'Gate'  : (chk(info['gate'],   passenger['gate']),   info['gate'],   passenger['gate']),
        'From'  : (chk(info['from'],   passenger['from']),   info['from'],   passenger['from']),
        'To'    : (chk(info['to'],     passenger['to']),     info['to'],     passenger['to']),
    }

    failed = [k for k, (icon, _, _) in field_checks.items() if icon == '❌']
    warned = [k for k, (icon, _, _) in field_checks.items() if icon == '⚠️']

    if not failed:
        status  = 'APPROVED'
        icon    = '✅'
        message = f"KIRISH RUXSAT ETILDI! Xush kelibsiz, {passenger['name']}!"
    else:
        status  = 'MISMATCH'
        icon    = '❌'
        message = f"Mos kelmadi: {', '.join(failed)}"

    return {
        'status'      : status,
        'icon'        : icon,
        'message'     : message,
        'match_type'  : match_type,
        'passenger'   : passenger,
        'field_checks': field_checks,
        'warnings'    : warned,
    }

print("✅ verify_passenger() tayyor!")

✅ verify_passenger() tayyor!


##  Step 8a — MATCH_TYPE_LABELS
Yo'lovchi topish usullarini o'zbek tilida ko'rsatish uchun lug'at

In [14]:
MATCH_TYPE_LABELS = {
    'exact_name'  : '🎯 Aniq mos (ism + flight)',
    'fuzzy_name'  : '🔁 Taxminiy mos (OCR xatosi hisobga olindi)',
    'seat_flight' : "💺 O'rindiq + flight bo'yicha topildi",
    'not_found'   : '❌ Topilmadi',
}

print('✅ MATCH_TYPE_LABELS tayyor!')

✅ MATCH_TYPE_LABELS tayyor!


##  Step 8b — format_result()
Tekshirish natijasini chiroyli matn ko'rinishida formatlaydi

In [15]:
def format_result(info, verification):
    """OCR va tekshirish natijasini o'qish uchun qulay matn shaklida qaytaradi."""
    sep  = '\u2550' * 48
    thin = '\u2500' * 48

    # --- Sarlavha ---
    out  = '\n' + sep + '\n'
    out += '  \u2708\ufe0f   BOARDING PASS VERIFICATION  \u2708\ufe0f\n'
    out += '  FRA (Frankfurt)  \u2192  TAS (Tashkent)\n'
    out += sep + '\n\n'

    # --- OCR o'qigan ma'lumotlar ---
    out += "\U0001f4cb  OCR O'QIGAN MA'LUMOTLAR\n"
    out += thin + '\n'
    out += f"  \U0001f464 Ism        : {info['name']}\n"
    out += f"  \u2708\ufe0f  Flight     : {info['flight']}\n"
    out += f"  \U0001f4c5 Sana       : {info['date']}\n"
    out += f"  \U0001f4ba O'rindiq   : {info['seat']}\n"
    out += f"  \U0001f6aa Gate       : {info['gate']}\n"
    out += f"  \U0001f6eb From       : {info['from']}\n"
    out += f"  \U0001f6ec To         : {info['to']}\n\n"

    # --- Bazadagi ma'lumotlar ---
    p = verification['passenger']
    if p is not None:
        out += "\U0001f5c4\ufe0f  BAZADAGI MA'LUMOTLAR\n"
        out += thin + '\n'
        out += f"  \U0001f464 Ism        : {p['name']}\n"
        out += f"  \U0001fae3 Pasport    : {p['passport']}\n"
        out += f"  \u2708\ufe0f  Flight     : {p['flight']}\n"
        out += f"  \U0001f4c5 Sana       : {p['date']}\n"
        out += f"  \U0001f4ba O'rindiq   : {p['seat']}\n"
        out += f"  \U0001f6aa Gate       : {p['gate']}\n"
        out += f"  \U0001f6eb From       : {p['from']}\n"
        out += f"  \U0001f6ec To         : {p['to']}\n"
        out += f"  \U0001f4cc Status     : {p['status']}\n"
        match_label = MATCH_TYPE_LABELS.get(verification['match_type'], verification['match_type'])
        out += f"  \U0001f50d Topish usuli: {match_label}\n\n"

        # --- Solishtiruv ---
        out += '\U0001f50d  SOLISHTIRUV NATIJALARI\n'
        out += thin + '\n'
        for field, (icon, ocr_val, db_val) in verification['field_checks'].items():
            if ocr_val.upper() == db_val.upper():
                out += f'  {icon} {field:<10}: {ocr_val}\n'
            else:
                out += f'  {icon} {field:<10}: OCR={ocr_val!r}  |  Baza={db_val!r}\n'
        out += '\n'

    # --- Asosiy natija ---
    out += sep + '\n'
    out += f"  {verification['icon']}  {verification['message']}\n"
    out += sep + '\n'

    if verification.get('warnings'):
        out += f"  \u26a0\ufe0f  OCR topa olmagan maydonlar: {', '.join(verification['warnings'])}\n"

    # --- OCR to'liq matn ---
    raw  = info['raw_text']
    out += "\n\U0001f4c4  OCR TO'LIQ MATN (birinchi 400 belgi):\n"
    out += thin + '\n'
    out += raw[:400]
    if len(raw) > 400:
        out += '...'
    out += '\n'
    return out

print('✅ format_result() tayyor!')

✅ format_result() tayyor!


##  Step 9 — scan_and_verify()
Gradio uchun asosiy funksiya: rasm kiradi → natija matni chiqadi

In [16]:
def scan_and_verify(image):
    """Pipeline: rasm → preprocessing → OCR → ma'lumot ajratish → tekshirish → natija"""
    if image is None:
        return "\u26a0\ufe0f Iltimos, boarding pass / ticket rasmini yuklang!"

    try:
        versions     = preprocess_advanced(image)          # 1. Preprocessing
        merged       = run_ocr_multi(*versions)            # 2. OCR
        info         = extract_ticket_info(merged)         # 3. Ma'lumot ajratish
        verification = verify_passenger(info)              # 4. Tekshirish
        return format_result(info, verification)           # 5. Formatlash
    except Exception as e:
        import traceback
        return f'❌ Xatolik:\n{str(e)}\n\n{traceback.format_exc()}'

print('✅ scan_and_verify() tayyor!')

✅ scan_and_verify() tayyor!


##  Step 10a — Gradio Yordamchi Funksiyalar
`get_passenger_table()` — baza jadvalini Gradio uchun qaytaradi  
`manual_verify()` — rasm yuklamasdan qo'lda test qilish uchun

In [17]:
def get_passenger_table():
    """Yo'lovchilar jadvalini Gradio Dataframe uchun qaytaradi."""
    display_df = df_passengers[['id', 'name', 'passport', 'seat', 'gate', 'status']].copy()
    display_df.columns = ['ID', 'Ism', 'Pasport', "O'rindiq", 'Gate', 'Holat']
    return display_df


def manual_verify(name, flight, seat, gate, date, frm, to):
    """Qo'lda kiritilgan ma'lumotlar bilan tekshirish (OCR o'tkazmasdan)."""
    info = {
        'raw_text': f'Manual: {name} {flight} {seat} {gate} {date} {frm} {to}',
        'name'    : name.strip()   or 'Not found',
        'flight'  : flight.strip() or 'Not found',
        'seat'    : seat.strip()   or 'Not found',
        'gate'    : gate.strip()   or 'Not found',
        'date'    : date.strip()   or 'Not found',
        'from'    : frm.strip()    or 'Not found',
        'to'      : to.strip()     or 'Not found',
    }
    verification = verify_passenger(info)
    return format_result(info, verification)

print('✅ Yordamchi funksiyalar tayyor!')

✅ Yordamchi funksiyalar tayyor!


##  Step 10b — Gradio UI (3 ta tab)
- **Tab 1:** Yo'lovchilar ro'yxati
- **Tab 2:** Ticket rasmini yuklash va tekshirish
- **Tab 3:** Qo'lda ma'lumot kiritib test qilish

In [18]:
with gr.Blocks(title='✈️ FRA → TAS Boarding Verification') as app:

    gr.Markdown("""
    # ✈️ Boarding Pass Verification System
    ### Flight HY502 — Frankfurt (FRA) → Tashkent (TAS) | 15 JUN 2025 | Gate B7
    """)

    # ── Tab 1: Yo'lovchilar ro'yxati ─────────────────────────────────
    with gr.Tab("📋 Yo'lovchilar Ro'yxati"):
        gr.Markdown("### HY502 parvoziga ro'yxatdan o'tgan yo'lovchilar")
        with gr.Row():
            with gr.Column(scale=3):
                passenger_table = gr.Dataframe(
                    value=get_passenger_table(),
                    label=f"Jami {len(df_passengers)} yo'lovchi",
                    interactive=False,
                    wrap=True,
                )
            with gr.Column(scale=1):
                gr.Markdown("""
                **Parvoz ma'lumotlari**
                - ✈️ Flight: **HY502**
                - 🛫 From: **FRA** (Frankfurt)
                - 🛬 To: **TAS** (Tashkent)
                - 📅 Sana: **15 JUN 2025**
                - 🕐 Jo'naydi: **13:40**
                - 🚪 Gate: **B7**

                **Holat belgilari**
                - CHECKED IN — ro'yxatdan o'tdi
                - BOARDING — posadka boshlandi
                """)
        refresh_btn = gr.Button('🔄 Yangilash', variant='secondary')
        refresh_btn.click(fn=get_passenger_table, outputs=passenger_table)

    # ── Tab 2: Ticket tekshirish ──────────────────────────────────────
    with gr.Tab('🎫 Ticket Tekshirish'):
        gr.Markdown("""
        ### Boarding pass / ticket rasmini yuklang
        Tizim OCR orqali barcha ma'lumotlarni o'qib, HY502 yo'lovchilar bazasi bilan solishtiradi.
        """)
        with gr.Row():
            with gr.Column(scale=1):
                image_input = gr.Image(
                    type='pil',
                    label='📤 Ticket / Boarding Pass yuklang',
                    height=350,
                )
                scan_btn = gr.Button('🔍 Tekshirish', variant='primary', size='lg')
                gr.Markdown("""
                **Qo'llab-quvvatlanadigan formatlar:**
                - 📄 Chop etilgan boarding pass
                - 📱 Elektron boarding pass (screenshot)
                - 🎫 Samolyot bileti (e-ticket)
                - 🌐 Barcha mamlakat standartlari
                """)
            with gr.Column(scale=1):
                result_output = gr.Textbox(
                    label='📋 Tekshirish Natijasi',
                    lines=32,
                )
        scan_btn.click(fn=scan_and_verify, inputs=image_input, outputs=result_output)
        image_input.upload(fn=scan_and_verify, inputs=image_input, outputs=result_output)

    # ── Tab 3: Manual test ────────────────────────────────────────────
    with gr.Tab('🧪 Manual Test'):
        gr.Markdown("""
        ### Rasm yuklamasdan ham test qilish
        OCR o'rniga to'g'ridan-to'g'ri ma'lumot kiriting.
        """)
        with gr.Row():
            with gr.Column():
                t_name   = gr.Textbox(label='Ism',    placeholder='KARIMOV BOBUR')
                t_flight = gr.Textbox(label='Flight', placeholder='HY502')
                t_seat   = gr.Textbox(label="O'rindiq", placeholder='12A')
                t_gate   = gr.Textbox(label='Gate',   placeholder='B7')
                t_date   = gr.Textbox(label='Sana',   placeholder='15 JUN 2025')
                t_frm    = gr.Textbox(label='From',   placeholder='ICN')
                t_to     = gr.Textbox(label='To',     placeholder='TAS')
                manual_btn = gr.Button('✅ Tekshirish', variant='primary')
            with gr.Column():
                manual_result = gr.Textbox(label='Natija', lines=30)
        manual_btn.click(
            fn=manual_verify,
            inputs=[t_name, t_flight, t_seat, t_gate, t_date, t_frm, t_to],
            outputs=manual_result,
        )

print('✅ Gradio UI tayyor! Keyingi katak: app.launch()')

✅ Gradio UI tayyor! Keyingi katak: app.launch()


## 🚀 Step 10c — Ilovani ishga tushirish
Brauzerda avtomatik ochiladi: http://localhost:7861

In [ ]:
app.launch(
    share=False,
    inbrowser=True,
    server_port=7861,
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


C:\Users\xolmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\xolmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 🧪 Step 11 — Gradiosiz Test
Ilovani ishga tushirmasdan, funksiyalarni to'g'ridan-to'g'ri test qilish

In [20]:
# Yo'lovchilar bazasini ko'rsatish
print('HY502 | FRA → TAS | 15 JUN 2025')
print('=' * 60)
print(df_passengers.to_string(index=False))

HY502 | FRA → TAS | 15 JUN 2025
  id                  name  passport flight from  to        date seat gate     status
P001      ISMOILOV AZIZBEK AA1234567  HY502  FRA TAS 15 JUN 2025  12A   B7 CHECKED IN
P002      RUSTAMOVA MADINA AA2345678  HY502  FRA TAS 15 JUN 2025  12B   B7 CHECKED IN
P003       SODIQOV SHERZOD AB3456789  HY502  FRA TAS 15 JUN 2025  14C   B7 CHECKED IN
P004      HAMIDOVA DILNOZA AB4567890  HY502  FRA TAS 15 JUN 2025  14D   B7   BOARDING
P005    MAMATQULOV ULUGBEK AC5678901  HY502  FRA TAS 15 JUN 2025  16A   B7 CHECKED IN
P006        SHARIPOV OYBEK AC6789012  HY502  FRA TAS 15 JUN 2025  16B   B7   BOARDING
P007  ABDURAHMONOVA NIGINA AD7890123  HY502  FRA TAS 15 JUN 2025  18C   B7 CHECKED IN
P008       JURAYEV ASADBEK AD8901234  HY502  FRA TAS 15 JUN 2025  18D   B7 CHECKED IN
P009       KAMILOV SHAHZOD AE9012345  HY502  FRA TAS 15 JUN 2025  20A   B7 CHECKED IN
P010       ERKINOVA MOHIRA AE0123456  HY502  FRA TAS 15 JUN 2025  20B   B7   BOARDING
P011      NABIYEV JAHO

In [21]:
# Test 1: Bazada BOR yo'lovchi — APPROVED bo'lishi kerak
test_pass = {
    'raw_text': 'BOARDING PASS XOLMUMINOV KUTBIDDIN HY502 FRA TAS 15 JUN 2025 SEAT 26C GATE B7',
    'name'    : 'XOLMUMINOV KUTBIDDIN',
    'flight'  : 'HY502',
    'date'    : '15 JUN 2025',
    'seat'    : '26C',
    'gate'    : 'B7',
    'from'    : 'FRA',
    'to'      : 'TAS',
}
v = verify_passenger(test_pass)
print(format_result(test_pass, v))


════════════════════════════════════════════════
  ✈️   BOARDING PASS VERIFICATION  ✈️
  FRA (Frankfurt)  →  TAS (Tashkent)
════════════════════════════════════════════════

📋  OCR O'QIGAN MA'LUMOTLAR
────────────────────────────────────────────────
  👤 Ism        : XOLMUMINOV KUTBIDDIN
  ✈️  Flight     : HY502
  📅 Sana       : 15 JUN 2025
  💺 O'rindiq   : 26C
  🚪 Gate       : B7
  🛫 From       : FRA
  🛬 To         : TAS

🗄️  BAZADAGI MA'LUMOTLAR
────────────────────────────────────────────────
  👤 Ism        : XOLMUMINOV QUTBIDDIN
  🫣 Pasport    : AH5678901
  ✈️  Flight     : HY502
  📅 Sana       : 15 JUN 2025
  💺 O'rindiq   : 26C
  🚪 Gate       : B7
  🛫 From       : FRA
  🛬 To         : TAS
  📌 Status     : CHECKED IN
  🔍 Topish usuli: 🔁 Taxminiy mos (OCR xatosi hisobga olindi)

🔍  SOLISHTIRUV NATIJALARI
────────────────────────────────────────────────
  ❌ Name      : OCR='XOLMUMINOV KUTBIDDIN'  |  Baza='XOLMUMINOV QUTBIDDIN'
  ✅ Flight    : HY502
  ✅ Date      : 15 JUN 2025
  ✅ Sea

In [22]:
# Test 2: Bazada YO'Q yo'lovchi — DENIED bo'lishi kerak
test_fail = {
    'raw_text': 'BOARDING PASS OROZOV ELBEK HY502 ICN TAS 15 JUN 2025 SEAT 30A GATE B7',
    'name'    : 'OROZOV ELBEK',
    'flight'  : 'HY502',
    'date'    : '15 JUN 2025',
    'seat'    : '30A',
    'gate'    : 'B7',
    'from'    : 'ICN',
    'to'      : 'TAS',
}
v2 = verify_passenger(test_fail)
print(format_result(test_fail, v2))


════════════════════════════════════════════════
  ✈️   BOARDING PASS VERIFICATION  ✈️
  FRA (Frankfurt)  →  TAS (Tashkent)
════════════════════════════════════════════════

📋  OCR O'QIGAN MA'LUMOTLAR
────────────────────────────────────────────────
  👤 Ism        : OROZOV ELBEK
  ✈️  Flight     : HY502
  📅 Sana       : 15 JUN 2025
  💺 O'rindiq   : 30A
  🚪 Gate       : B7
  🛫 From       : ICN
  🛬 To         : TAS

════════════════════════════════════════════════
  ❌  Yo'lovchi bazada topilmadi!
════════════════════════════════════════════════

📄  OCR TO'LIQ MATN (birinchi 400 belgi):
────────────────────────────────────────────────
BOARDING PASS OROZOV ELBEK HY502 ICN TAS 15 JUN 2025 SEAT 30A GATE B7



In [23]:
# Test 3: OCR xatosi bilan — fuzzy matching ishlashi kerak
test_fuzzy = {
    'raw_text': 'XOLMUMINOV KUTBIDDIN HY5O2 FRA TAS',  # 0 vs O xatosi
    'name'    : 'Xolmuminov Kutbiddin',
    'flight'  : 'HY502',
    'date'    : 'Not found',
    'seat'    : '26C',
    'gate'    : 'B7',
    'from'    : 'FRA',
    'to'      : 'TAS',
}
v3 = verify_passenger(test_fuzzy)
print(f"Topish usuli: {v3['match_type']}")
print(format_result(test_fuzzy, v3))

Topish usuli: fuzzy_name

════════════════════════════════════════════════
  ✈️   BOARDING PASS VERIFICATION  ✈️
  FRA (Frankfurt)  →  TAS (Tashkent)
════════════════════════════════════════════════

📋  OCR O'QIGAN MA'LUMOTLAR
────────────────────────────────────────────────
  👤 Ism        : Xolmuminov Kutbiddin
  ✈️  Flight     : HY502
  📅 Sana       : Not found
  💺 O'rindiq   : 26C
  🚪 Gate       : B7
  🛫 From       : FRA
  🛬 To         : TAS

🗄️  BAZADAGI MA'LUMOTLAR
────────────────────────────────────────────────
  👤 Ism        : XOLMUMINOV QUTBIDDIN
  🫣 Pasport    : AH5678901
  ✈️  Flight     : HY502
  📅 Sana       : 15 JUN 2025
  💺 O'rindiq   : 26C
  🚪 Gate       : B7
  🛫 From       : FRA
  🛬 To         : TAS
  📌 Status     : CHECKED IN
  🔍 Topish usuli: 🔁 Taxminiy mos (OCR xatosi hisobga olindi)

🔍  SOLISHTIRUV NATIJALARI
────────────────────────────────────────────────
  ❌ Name      : OCR='Xolmuminov Kutbiddin'  |  Baza='XOLMUMINOV QUTBIDDIN'
  ✅ Flight    : HY502
  ⚠️ Date   

In [24]:
from PIL import Image, ImageDraw, ImageFont
import os

def create_boarding_pass_pass():
    W, H = 700, 330
    img = Image.new('RGB', (W, H), color='white')
    d   = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype("arial.ttf", 22)
        font_name  = ImageFont.truetype("arial.ttf", 38)
        font_big   = ImageFont.truetype("arial.ttf", 28)
        font_med   = ImageFont.truetype("arial.ttf", 18)
        font_sm    = ImageFont.truetype("arial.ttf", 13)
    except:
        font_title = font_name = font_big = font_med = font_sm = ImageFont.load_default()

    BLUE = '#003366'
    GRAY = '#888888'

    # --- Header: filled rect only, NO outer border (border generates garbage OCR) ---
    d.rectangle([0, 0, W, 55], fill=BLUE)
    d.text((15, 15),  "BOARDING PASS",   fill='white', font=font_title)
    d.text((450, 18), "UZAIRWAYS / HY",  fill='white', font=font_sm)
    d.rectangle([0, 55, W, 58], fill='#ff8c00')  # thin accent only

    # --- PASSENGER: single keyword label, value directly below ---
    d.text((15, 65),  "PASSENGER",      fill=GRAY,    font=font_sm)
    d.text((15, 80),  "XOLMUMINOV QUTBIDDIN",  fill='black', font=font_name)

    d.rectangle([0, 133, W, 134], fill='#eeeeee')  # thin divider

    # --- ROUTE: "ICN > TAS" drawn as ONE string ---
    # OCR reads it as single block, route regex r'([A-Z]{3})\s*[>]+\s*([A-Z]{3})' matches
    d.text((15, 142),  "FROM",          fill=GRAY,    font=font_sm)
    d.text((15, 158),  "FRA > TAS",     fill=BLUE,    font=font_big)

    # --- FLIGHT ---
    d.text((260, 142), "FLIGHT",        fill=GRAY,    font=font_sm)
    d.text((260, 158), "HY502",         fill='black', font=font_big)

    # --- SEAT ---
    d.text((460, 142), "SEAT",          fill=GRAY,    font=font_sm)
    d.text((460, 158), "26C",           fill='black', font=font_big)

    # --- GATE: same row as SEAT/FLIGHT (no DATE nearby — no more 'GATE 15' mistake) ---
    d.text((570, 142), "GATE",          fill=GRAY,    font=font_sm)
    d.text((570, 158), "B7",            fill='black', font=font_big)

    d.rectangle([0, 200, W, 201], fill='#eeeeee')

    # --- DATE: separate row, far from GATE ---
    d.text((15, 208),  "DATE",          fill=GRAY,    font=font_sm)
    d.text((15, 223),  "15 JUN 2025",   fill='black', font=font_med)

    # --- PASSPORT ---
    d.text((230, 208), "PASSPORT",      fill=GRAY,    font=font_sm)
    d.text((230, 223), "AH5678901",     fill='black', font=font_med)

    # --- CLASS ---
    d.text((460, 208), "CLASS",         fill=GRAY,    font=font_sm)
    d.text((460, 223), "ECONOMY",       fill='black', font=font_med)

    d.rectangle([0, 258, W, 259], fill='#eeeeee')

    # --- STATUS ---
    d.text((15,  267), "STATUS CHECKED IN", fill='#006600', font=font_med)
    d.text((410, 267), "DEPARTS 1340",       fill=GRAY,      font=font_med)

    path = 'ticket_PASS_XOLMUMINOV_v5.png'
    img.save(path)
    print(f"✅ Saqlandi: {os.path.abspath(path)}")
    img.show()

create_boarding_pass_pass()


✅ Saqlandi: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\Computer Vision 2\4 dars\ticket_PASS_XOLMUMINOV_v5.png


In [25]:
from PIL import Image, ImageDraw, ImageFont
import os

def create_boarding_pass_fail():
    W, H = 800, 360
    img  = Image.new('RGB', (W, H), color='white')
    d    = ImageDraw.Draw(img)

    try:
        font_big  = ImageFont.truetype("arial.ttf", 28)
        font_med  = ImageFont.truetype("arial.ttf", 20)
        font_sm   = ImageFont.truetype("arial.ttf", 15)
    except:
        font_big  = ImageFont.load_default()
        font_med  = font_big
        font_sm   = font_big

    # Border
    d.rectangle([0, 0, W-1, H-1], outline='black', width=3)
    d.rectangle([0, 0, W-1, 55],  fill='#8B0000')  # Dark red — fail

    # Header
    d.text((20, 12),  "BOARDING PASS",       fill='white', font=font_big)
    d.text((500, 16), "UZAIRWAYS / HY",      fill='white', font=font_med)

    # Route — same flight
    d.text((20,  75),  "FROM",      fill='gray',  font=font_sm)
    d.text((200, 75),  "TO",        fill='gray',  font=font_sm)
    d.text((20,  95),  "FRA",       fill='black', font=font_big)
    d.text((200, 95),  "TAS",       fill='black', font=font_big)
    d.text((20,  130), "INCHEON",   fill='gray',  font=font_sm)
    d.text((200, 130), "TASHKENT",  fill='gray',  font=font_sm)
    d.text((130, 100), u"\u2192",   fill='#8B0000', font=font_big)

    # Different passenger — NOT in database
    d.text((20,  165), "PASSENGER NAME", fill='gray',  font=font_sm)
    d.text((20,  183), "KONRAD ADEL",     fill='black', font=font_big)

    # Same flight, same gate (sneaky — flight exists but person doesn't)
    d.text((400, 75),  "FLIGHT",         fill='gray',  font=font_sm)
    d.text((400, 93),  "HY502",          fill='black', font=font_big)

    d.text((400, 135), "DATE",           fill='gray',  font=font_sm)
    d.text((400, 153), "15 JUN 2025",    fill='black', font=font_med)

    d.text((600, 75),  "SEAT",           fill='gray',  font=font_sm)
    d.text((600, 93),  "30C",            fill='black', font=font_big)  # Seat not in DB

    d.text((600, 135), "GATE",           fill='gray',  font=font_sm)
    d.text((600, 153), "B7",             fill='black', font=font_big)

    # Different passport
    d.text((20,  240), "PASSPORT",       fill='gray',  font=font_sm)
    d.text((20,  258), "KR9876543",      fill='black', font=font_med)

    d.text((400, 240), "CLASS",          fill='gray',  font=font_sm)
    d.text((400, 258), "ECONOMY",        fill='black', font=font_med)

    d.text((600, 240), "TERMINAL",       fill='gray',  font=font_sm)
    d.text((600, 258), "T2",             fill='black', font=font_med)

    # Barcode placeholder
    d.rectangle([20, 305, 780, 345], outline='black', width=1)
    for i in range(20, 780, 6):
        if i % 10 < 5:
            d.rectangle([i, 306, i+4, 344], fill='black')
    d.text((330, 348), "HY502-KIMJINWOO-30C-B7", fill='gray', font=font_sm)

    path = 'ticket_FAIL_KONRAD_ADEL.png'
    img.save(path)
    print(f"✅ Saqlandi: {os.path.abspath(path)}")
    img.show()

create_boarding_pass_fail()


✅ Saqlandi: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\Computer Vision 2\4 dars\ticket_FAIL_KONRAD_ADEL.png


C:\Users\xolmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\xolmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\xolmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
